In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "distilbert-base-uncased"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
max_length = 128
l2_normalize = True
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "l2_normalize": l2_normalize,
    "seed": seed,
})

{'model_name': 'distilbert-base-uncased', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'batch_size': 64, 'max_length': 128, 'l2_normalize': True, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

{'num_examples': 1500, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   

                                  sentence2  label  
0      A man wearing a hard hat is dancing.   5.00  
1                A child is riding a horse.   4.75  
2  The man is feeding a mouse to the snake.   5.00  
3                  A man is playing guitar.   2.40  
4                 A man is playing a flute.   2.75  


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print(model_name)
print(type(model).__name__)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
DistilBertModel


In [4]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

enc_len1 = tokenizer(sentences1, truncation=True, max_length=max_length, add_special_tokens=True)
enc_len2 = tokenizer(sentences2, truncation=True, max_length=max_length, add_special_tokens=True)

len1 = np.array([len(x) for x in enc_len1["input_ids"]], dtype=np.int32)
len2 = np.array([len(x) for x in enc_len2["input_ids"]], dtype=np.int32)
pair_total_length = len1 + len2
pair_mean_length = (len1 + len2) / 2.0

length_threshold = float(np.median(pair_total_length))
length_bucket = np.where(pair_total_length <= length_threshold, "short", "long")

df["token_length_s1"] = len1
df["token_length_s2"] = len2
df["pair_total_length"] = pair_total_length
df["pair_mean_length"] = pair_mean_length
df["length_bucket"] = length_bucket

token_stats = {
    "s1_mean": float(len1.mean()),
    "s1_std": float(len1.std()),
    "s1_min": int(len1.min()),
    "s1_max": int(len1.max()),
    "s2_mean": float(len2.mean()),
    "s2_std": float(len2.std()),
    "s2_min": int(len2.min()),
    "s2_max": int(len2.max()),
    "pair_total_mean": float(pair_total_length.mean()),
    "pair_total_std": float(pair_total_length.std()),
    "pair_total_median": float(np.median(pair_total_length)),
    "short_long_threshold_total_tokens": length_threshold,
    "short_count": int((length_bucket == "short").sum()),
    "long_count": int((length_bucket == "long").sum()),
}

print(token_stats)
print(df[["sentence1", "sentence2", "token_length_s1", "token_length_s2", "pair_total_length", "length_bucket"]].head(10))

{'s1_mean': 16.235333333333333, 's1_std': 7.445084612607763, 's1_min': 5, 's1_max': 45, 's2_mean': 16.237333333333332, 's2_std': 7.470988749080599, 's2_min': 6, 's2_max': 53, 'pair_total_mean': 32.47266666666667, 'pair_total_std': 14.173540591146903, 'pair_total_median': 29.0, 'short_long_threshold_total_tokens': 29.0, 'short_count': 780, 'long_count': 720}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  token_length_s1  token_length_s2  \
0      A man wearing a hard hat is dancing.               11               11   
1   

In [5]:
def encode_cls_embeddings(texts, tokenizer, model, device, batch_size=32, max_length=128, l2_normalize=True):
    all_embeddings = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            batch = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            hidden = outputs.last_hidden_state
            cls_embeddings = hidden[:, 0, :]
            if l2_normalize:
                cls_embeddings = F.normalize(cls_embeddings, p=2, dim=1)
            all_embeddings.append(cls_embeddings.detach().cpu())
    return torch.cat(all_embeddings, dim=0).numpy()

emb1 = encode_cls_embeddings(
    sentences1,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=batch_size,
    max_length=max_length,
    l2_normalize=l2_normalize,
)

emb2 = encode_cls_embeddings(
    sentences2,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=batch_size,
    max_length=max_length,
    l2_normalize=l2_normalize,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1) if l2_normalize else np.sum(
    emb1 * emb2, axis=1
) / (
    np.linalg.norm(emb1, axis=1) * np.linalg.norm(emb2, axis=1) + 1e-12
)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

print({
    "embedding_shape_s1": tuple(emb1.shape),
    "embedding_shape_s2": tuple(emb2.shape),
    "cosine_min": float(cosine_similarity.min()),
    "cosine_max": float(cosine_similarity.max()),
})

{'embedding_shape_s1': (1500, 768), 'embedding_shape_s2': (1500, 768), 'cosine_min': 0.7878332138061523, 'cosine_max': 0.9999999403953552}


In [6]:
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])
results_df["signed_error"] = results_df["predicted_score_0_5"] - results_df["label"]

score_mean = float(np.mean(predicted_score_0_5))
score_std = float(np.std(predicted_score_0_5))
label_mean = float(np.mean(labels))
label_std = float(np.std(labels))
mae = float(np.mean(np.abs(predicted_score_0_5 - labels)))
rmse = float(np.sqrt(np.mean((predicted_score_0_5 - labels) ** 2)))

print(results_df[[
    "sentence1", "sentence2", "label", "token_length_s1", "token_length_s2",
    "pair_total_length", "length_bucket", "cosine_similarity", "predicted_score_0_5",
    "absolute_error", "signed_error"
]].head(10))

                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  label  token_length_s1  \
0      A man wearing a hard hat is dancing.  5.000               11   
1                A child is riding a horse.  4.750               10   
2  The man is feeding a mouse to the snake.  5.000               12   
3                  A man is playing guitar.  2.400                9   
4                 A man is playing a flute.  2.750                9   
5                  A man is cutting onions.  2.615                9   
6       The man is erasing th

In [7]:
group_rows = []
for bucket, group in results_df.groupby("length_bucket"):
    bucket_pred = group["predicted_score_0_5"].to_numpy(dtype=np.float32)
    bucket_label = group["label"].to_numpy(dtype=np.float32)
    group_rows.append({
        "length_bucket": bucket,
        "count": int(len(group)),
        "pearson": float(pearsonr(bucket_pred, bucket_label).statistic),
        "spearman": float(spearmanr(bucket_pred, bucket_label).statistic),
        "mae": float(np.mean(np.abs(bucket_pred - bucket_label))),
        "rmse": float(np.sqrt(np.mean((bucket_pred - bucket_label) ** 2))),
        "pred_mean": float(bucket_pred.mean()),
        "label_mean": float(bucket_label.mean()),
        "pair_total_length_mean": float(group["pair_total_length"].mean()),
    })

length_metrics_df = pd.DataFrame(group_rows).sort_values("length_bucket").reset_index(drop=True)
print(length_metrics_df.to_string(index=False))

length_bucket  count  pearson  spearman      mae     rmse  pred_mean  label_mean  pair_total_length_mean
         long    720 0.555853  0.600850 2.530819 2.868872   4.849115    2.319903               44.579167
        short    780 0.384170  0.405208 2.480381 2.918139   4.876645    2.404527               21.297436


In [8]:
top_k = 5

error_tables = {}
for bucket in ["short", "long"]:
    bucket_df = results_df[results_df["length_bucket"] == bucket].copy()
    worst = bucket_df.nlargest(top_k, "absolute_error")[[
        "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity",
        "absolute_error", "signed_error", "token_length_s1", "token_length_s2", "pair_total_length"
    ]].reset_index(drop=True)
    best = bucket_df.nsmallest(top_k, "absolute_error")[[
        "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity",
        "absolute_error", "signed_error", "token_length_s1", "token_length_s2", "pair_total_length"
    ]].reset_index(drop=True)
    error_tables[bucket] = {"worst": worst, "best": best}

for bucket in ["short", "long"]:
    print(f"Length bucket: {bucket} | Worst absolute errors")
    print(error_tables[bucket]["worst"].to_string(index=False))
    print()
    print(f"Length bucket: {bucket} | Best absolute errors")
    print(error_tables[bucket]["best"].to_string(index=False))
    print()

Length bucket: short | Worst absolute errors
                                      sentence1                         sentence2  label  predicted_score_0_5  cosine_similarity  absolute_error  signed_error  token_length_s1  token_length_s2  pair_total_length
                  A woman is riding on a horse.       A man is shooting off guns.    0.0             4.937448           0.974979        4.937448      4.937448               10                9                 19
                     A man is cutting a potato.  A woman is climbing a rock wall.    0.0             4.932104           0.972842        4.932104      4.932104                9               10                 19
                    A woman is applying makeup.        A man is playing a guitar.    0.0             4.930091           0.972037        4.930091      4.930091                8                9                 17
             A cat is pouncing on a trampoline.        A man is slicing a tomato.    0.0             4.9247

In [9]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pooling: CLS")
print(f"inference_type: raw_transformers_manual_tokenization")
print(f"l2_normalize: {l2_normalize}")
print(f"max_length: {max_length}")
print(f"batch_size: {batch_size}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"mae: {mae:.6f}")
print(f"rmse: {rmse:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"sentence1_token_mean: {token_stats['s1_mean']:.6f}")
print(f"sentence2_token_mean: {token_stats['s2_mean']:.6f}")
print(f"pair_total_token_mean: {token_stats['pair_total_mean']:.6f}")
print(f"pair_total_token_median: {token_stats['pair_total_median']:.6f}")
print(f"short_long_threshold_total_tokens: {token_stats['short_long_threshold_total_tokens']:.6f}")
print(f"short_count: {token_stats['short_count']}")
print(f"long_count: {token_stats['long_count']}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

device_used: mps
model_name: distilbert-base-uncased
dataset_split: glue/stsb/validation
num_examples: 1500
pooling: CLS
inference_type: raw_transformers_manual_tokenization
l2_normalize: True
max_length: 128
batch_size: 64
spearman_correlation: 0.486273
pearson_correlation: 0.460718
mae: 2.504591
rmse: 2.894595
predicted_score_mean: 4.863431
predicted_score_std: 0.092162
label_mean: 2.363908
label_std: 1.499985
sentence1_token_mean: 16.235333
sentence2_token_mean: 16.237333
pair_total_token_mean: 32.472667
pair_total_token_median: 29.000000
short_long_threshold_total_tokens: 29.000000
short_count: 780
long_count: 720
runtime_seconds: 5.84
